In [1]:
import os
import gc
import re
import zipfile
import getpass
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import cohen_kappa_score

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from arabert.preprocess import ArabertPreprocessor

import scipy.optimize as optimize
from functools import partial

# ----------------------------------------------------------
# CORAL Trainer
# ----------------------------------------------------------
class CORALTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        device = logits.device

        target = torch.zeros_like(logits, device=device)
        for k in range(logits.shape[1]):
            target[:, k] = (labels > k).float()

        loss_per_sample = F.binary_cross_entropy_with_logits(
            logits, target, reduction='none'
        ).mean(dim=1)

        if self.class_weights is not None:
            w = self.class_weights.to(device)[labels]
            loss = (loss_per_sample * w).mean()
        else:
            loss = loss_per_sample.mean()

        return (loss, outputs) if return_outputs else loss


def compute_metrics_coral(eval_pred):
    logits, labels = eval_pred
    pred_labels = (logits > 0).sum(axis=1)
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    return {"qwk": qwk}


MODEL_NAME = "aubmindlab/bert-base-arabertv02"
PROCESSED_DIR = '../data/processed/'
N_SPLITS = 5

In [2]:
arabert_prep = ArabertPreprocessor(model_name=MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence_Normalized"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

print("Loading training data...")
df_all = pd.read_parquet(PROCESSED_DIR + 'train_cleaned.parquet')
df_all['label'] = df_all['label'].astype(np.int64)
print(f"Training set size: {len(df_all)} sentences")

Loading training data...
Training set size: 54626 sentences


In [3]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_true_labels = np.zeros(len(df_all), dtype=int)
seed_results = {}

for current_seed in [42, 123]:
    print(f"\n{'='*60}")
    print(f"SEED: {current_seed}")
    print(f"{'='*60}")

    current_oof_logits = np.zeros((len(df_all), 18))

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(df_all, df_all['label'])
    ):
        print(f"\n--- Fold {fold + 1}/{N_SPLITS} (seed {current_seed}) ---")

        train_fold = df_all.iloc[train_idx]
        val_fold   = df_all.iloc[val_idx]

        # Class weights (computed per fold on train split only)
        cw = compute_class_weight(
            'balanced', classes=np.arange(19), y=train_fold['label'].values
        )
        cw_normalized = np.clip(cw, 0.5, 5.0)
        cw_normalized = cw_normalized / cw_normalized.mean()
        class_weights_tensor = torch.tensor(cw_normalized, dtype=torch.float32)

        # HF Datasets
        hf_train = Dataset.from_pandas(train_fold[['Sentence_Normalized', 'label']])
        hf_val   = Dataset.from_pandas(val_fold[['Sentence_Normalized', 'label']])
        tok_train = hf_train.map(tokenize_function, batched=True)
        tok_val   = hf_val.map(tokenize_function, batched=True)
        tok_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
        tok_val.set_format(type='torch',   columns=['input_ids', 'attention_mask', 'label'])

        # Model
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=18
        )
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
            lora_dropout=0.05, target_modules=["query", "value", "dense"]
        )
        model = get_peft_model(model, lora_config)

        training_args = TrainingArguments(
            output_dir=f"../saved_models/arabert_coral_fold{fold+1}_seed{current_seed}",
            eval_strategy="epoch", save_strategy="epoch",
            learning_rate=3e-4, per_device_train_batch_size=8,
            gradient_accumulation_steps=4, per_device_eval_batch_size=16,
            num_train_epochs=5, weight_decay=0.01,
            load_best_model_at_end=True, metric_for_best_model="qwk",
            greater_is_better=True, bf16=True, fp16=False, seed=current_seed,
        )

        trainer = CORALTrainer(
            model=model, args=training_args,
            train_dataset=tok_train, eval_dataset=tok_val,
            compute_metrics=compute_metrics_coral,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
            class_weights=class_weights_tensor,
        )
        trainer.train()

        # Collect OOF logits
        val_pred = trainer.predict(tok_val)
        current_oof_logits[val_idx] = val_pred.predictions[:, :18]
        if current_seed == 42:
            oof_true_labels[val_idx] = val_pred.label_ids.astype(int)

        # NOTE: blind test is NOT predicted here.
        # Fold model is saved to disk; we reload the best checkpoint in Cell 5.

        del model, trainer, training_args
        gc.collect()
        torch.cuda.empty_cache()

    seed_results[current_seed] = {'oof_logits': current_oof_logits}
    oof_qwk = cohen_kappa_score(
        oof_true_labels, (current_oof_logits > 0).sum(axis=1), weights='quadratic'
    )
    print(f"\nSeed {current_seed} OOF QWK (delta=0): {oof_qwk:.4f}")

print("\nAll folds finished. Proceeding to post-processing on OOF.")


SEED: 42

--- Fold 1/5 (seed 42) ---


Map:   0%|          | 0/43700 [00:00<?, ? examples/s]

Map:   0%|          | 0/10926 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/6825 [00:00<?, ?it/s]

...

{'eval_loss': 0.05981225147843361, 'eval_qwk': 0.8273218752915688, 'eval_runtime': 45.9151, 'eval_samples_per_second': 237.939, 'eval_steps_per_second': 14.875, 'epoch': 4.0}
{'loss': 0.0399, 'grad_norm': 0.4308106005191803, 'learning_rate': 5.824175824175824e-05, 'epoch': 4.03}
{'loss': 0.0358, 'grad_norm': 0.5656532645225525, 'learning_rate': 3.626373626373626e-05, 'epoch': 4.39}
{'loss': 0.037, 'grad_norm': 0.49962472915649414, 'learning_rate': 1.4285714285714284e-05, 'epoch': 4.76}


  0%|          | 0/683 [00:00<?, ?it/s]

{'eval_loss': 0.06116168573498726, 'eval_qwk': 0.8342324361811051, 'eval_runtime': 46.0434, 'eval_samples_per_second': 237.276, 'eval_steps_per_second': 14.834, 'epoch': 5.0}
{'train_runtime': 2989.528, 'train_samples_per_second': 73.09, 'train_steps_per_second': 2.283, 'train_loss': 0.050967052725208545, 'epoch': 5.0}


  0%|          | 0/683 [00:00<?, ?it/s]


Seed 123 OOF QWK (delta=0): 0.8373

All folds finished. Proceeding to post-processing on OOF.


In [ ]:
oof_logits_seed42  = seed_results[42]['oof_logits']
oof_logits_seed123 = seed_results[123]['oof_logits']

# ----------------------------------------------------------
# Step 4a: Find optimal blend ratio on OOF
# ----------------------------------------------------------
print("Step 4a: Searching for optimal seed blend ratio on OOF...")
best_w, best_qwk_blend = 0.5, -1.0

for w in np.arange(0.0, 1.05, 0.05):
    temp_blend = w * oof_logits_seed42 + (1 - w) * oof_logits_seed123
    preds = (temp_blend > 0).sum(axis=1)   # delta=0 baseline for ratio search
    qwk = cohen_kappa_score(oof_true_labels, preds, weights='quadratic')
    if qwk > best_qwk_blend:
        best_qwk_blend = qwk
        best_w = w

print(f"  Optimal ratio: {best_w*100:.0f}% Seed42 + {(1-best_w)*100:.0f}% Seed123")
blended_oof_logits = best_w * oof_logits_seed42 + (1 - best_w) * oof_logits_seed123

# ----------------------------------------------------------
# Step 4b: Optimize 18 thresholds via Nelder-Mead on blended OOF
# ----------------------------------------------------------
print("\nStep 4b: Optimizing 18 thresholds (Nelder-Mead) on blended OOF...")

def kappa_loss(coef, X, y):
    preds = (X > coef).sum(axis=1)
    return -cohen_kappa_score(y, preds, weights='quadratic')

result = optimize.minimize(
    partial(kappa_loss, X=blended_oof_logits, y=oof_true_labels),
    x0=np.zeros(18),
    method='nelder-mead',
    options={'xatol': 1e-4, 'fatol': 1e-4, 'maxiter': 50000},
)
best_thresholds = result['x']
oof_qwk_final   = -result.fun

print(f"  OOF QWK (blended, Nelder-Mead): {oof_qwk_final:.4f}")
print(f"  18 thresholds: {np.round(best_thresholds, 3)}")
print("\n[FROZEN] blend weight:", round(best_w, 2))
print("[FROZEN] thresholds:   ", np.round(best_thresholds, 3))
print("\nAll hyperparameters finalized on OOF. Ready for blind test inference.")

Step 4a: Searching for optimal seed blend ratio on OOF...
  Optimal ratio: 50% Seed42 + 50% Seed123

Step 4b: Optimizing 18 thresholds (Nelder-Mead) on blended OOF...
  OOF QWK (blended, Nelder-Mead): 0.8417
  18 thresholds: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

[FROZEN] blend weight: 0.5
[FROZEN] thresholds:    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

All hyperparameters finalized on OOF. Ready for blind test inference.


In [6]:
print("Please enter your Hugging Face Access Token:")
access_token = getpass.getpass("Token: ")

print("Downloading blind test dataset...")
barec_sent = load_dataset(
    "CAMeL-Lab/BAREC-Shared-Task-2026-BlindTest-sent", token=access_token
)
df_blind_test = barec_sent['train'].to_pandas()

print("Preprocessing blind test sentences...")
df_blind_test['Sentence_Normalized'] = df_blind_test['Sentence'].apply(
    lambda x: arabert_prep.preprocess(str(x))
)

hf_blind_test = Dataset.from_pandas(df_blind_test[['Sentence_Normalized']])
tokenized_blind_test = hf_blind_test.map(tokenize_function, batched=True)
tokenized_blind_test.set_format(type='torch', columns=['input_ids', 'attention_mask'])

print(f"Blind test size: {len(df_blind_test)} sentences")

import os

def get_best_checkpoint_dir(output_dir):
    state_path = os.path.join(output_dir, "trainer_state.json")
    if os.path.exists(state_path):
        with open(state_path) as f:
            state = json.load(f)
        best_ckpt = state.get("best_model_checkpoint")
        if best_ckpt and os.path.isdir(best_ckpt):
            return best_ckpt
    # Fallback: pick the checkpoint-<N> subfolder with the highest step
    ckpt_dirs = sorted(
        [
            d for d in os.listdir(output_dir)
            if d.startswith("checkpoint-") and
               os.path.isfile(os.path.join(output_dir, d, "adapter_config.json"))
        ],
        key=lambda x: int(x.split("-")[1]),
    )
    if not ckpt_dirs:
        raise FileNotFoundError(
            f"No valid PEFT checkpoint found in {output_dir}. "
            "Expected subfolders containing adapter_config.json."
        )
    return os.path.join(output_dir, ckpt_dirs[-1])


from peft import PeftModel

test_logits_per_seed = {}

for current_seed in [42, 123]:
    print(f"\nInferring blind test — seed {current_seed}...")
    fold_test_logits = []

    for fold in range(N_SPLITS):
        output_dir = f"../saved_models/arabert_coral_fold{fold+1}_seed{current_seed}"
        ckpt_dir   = get_best_checkpoint_dir(output_dir)
        print(f"  Fold {fold+1}: loading adapter from {ckpt_dir}")

        # Step 1 — base backbone
        base_model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=18
        )
        # Step 2 — attach LoRA adapter from the correct subfolder
        peft_model = PeftModel.from_pretrained(base_model, ckpt_dir)
        # Step 3 — merge & unload for fast inference
        model = peft_model.merge_and_unload()

        infer_args = TrainingArguments(
            output_dir="/tmp/infer",
            per_device_eval_batch_size=32,
            bf16=True, fp16=False, seed=current_seed,
        )
        infer_trainer = Trainer(model=model, args=infer_args)
        pred = infer_trainer.predict(tokenized_blind_test)
        fold_test_logits.append(pred.predictions[:, :18])

        del model, peft_model, base_model, infer_trainer, infer_args
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  Fold {fold+1} done.")

    test_logits_per_seed[current_seed] = np.mean(fold_test_logits, axis=0)

# Apply the FROZEN blend weight to test logits
blended_test_logits = (
    best_w * test_logits_per_seed[42] +
    (1 - best_w) * test_logits_per_seed[123]
)

print("\nBlind test inference complete. Proceeding to final submission.")


Please enter your Hugging Face Access Token:
Preprocessing blind test sentences...


Map:   0%|          | 0/8077 [00:00<?, ? examples/s]

Blind test size: 8077 sentences

Inferring blind test — seed 42...
  Fold 1: loading adapter from ../saved_models/arabert_coral_fold1_seed42\checkpoint-6825


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/253 [00:00<?, ?it/s]

...

  0%|          | 0/253 [00:00<?, ?it/s]

  Fold 4 done.
  Fold 5: loading adapter from ../saved_models/arabert_coral_fold5_seed123\checkpoint-6825


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/253 [00:00<?, ?it/s]

  Fold 5 done.

Blind test inference complete. Proceeding to final submission.


In [7]:
assert oof_true_labels.min() == 0 and oof_true_labels.max() == 18, (
    f"Unexpected label range in OOF: [{oof_true_labels.min()}, {oof_true_labels.max()}]. "
    "Expected [0, 18]. Check parquet encoding before proceeding."
)
print(f"Label check passed: OOF range [{oof_true_labels.min()}, {oof_true_labels.max()}] → submission range [1, 19]")

test_ids = df_blind_test['ID'].values

# +1 converts to submission  → range 1..19
final_preds = (blended_test_logits > best_thresholds).sum(axis=1) + 1
final_preds = np.clip(final_preds, 1, 19)

submission_df = pd.DataFrame({'Sentence ID': test_ids, 'Prediction': final_preds})
submission_df.to_csv('prediction', index=False, lineterminator='\n')

with zipfile.ZipFile('prediction_final.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('prediction', arcname='prediction')

print("Submission generated: prediction_final.zip")
print(f"OOF QWK (Nelder-Mead, blended): {oof_qwk_final:.4f}")
print(submission_df['Prediction'].value_counts().sort_index())

Label check passed: OOF range [0, 18] → submission range [1, 19]
Submission generated: prediction_final.zip
OOF QWK (Nelder-Mead, blended): 0.8417
Prediction
1       29
2       63
3      164
4      182
5      284
6      363
7      483
8      767
9      549
10    1465
11     712
12    1259
13     567
14     734
15     287
16     154
17      15
Name: count, dtype: int64
